Lab 24.2 – Save & Load Models
Goal: Train a pipeline, persist it with joblib, reload it later, and run predictions exactly as before. You’ll also save metadata (versioning), create a tiny model card, and test integrity.

We’ll use the Titanic classification pipeline from the previous lab.

In [1]:
# pip install scikit-learn pandas numpy seaborn joblib
import numpy as np, pandas as pd, seaborn as sns, joblib, json, hashlib, sys, sklearn, platform, datetime
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
np.random.seed(0)

1) Train a pipeline (preprocessing + model)

In [2]:
df = sns.load_dataset('titanic').drop(columns=['alive'])
target = 'survived'
features = ['pclass','sex','age','sibsp','parch','fare','embarked','class','who','alone']
df = df[features+[target]].copy()
df[target] = df[target].astype(int)

X = df[features]; y = df[target]
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

pipe = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=2000, random_state=0))])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipe.fit(Xtr, ytr)

proba = pipe.predict_proba(Xte)[:,1]
print("Hold‑out Accuracy:", round(accuracy_score(yte, pipe.predict(Xte)),3))
print("Hold‑out ROC‑AUC:", round(roc_auc_score(yte, proba),3))

Hold‑out Accuracy: 0.832
Hold‑out ROC‑AUC: 0.869


2) Save the model with joblib.dump
Create a small artifact bundle: model file + metadata JSON + (optional) training columns.

In [6]:
MODEL_PATH = "titanic_pipe.joblib"
META_PATH  = "titanic_meta.json"

# 2.1 save pipeline
joblib.dump(pipe, MODEL_PATH)

import datetime

# 2.2 metadata for reproducibility
meta = {
   "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat().replace("+00:00", "Z"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "sklearn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "model_file": MODEL_PATH,
    "target": target,
    "features": features,
    "train_rows": int(len(Xtr)),
    "notes": "Titanic pipeline: impute+scale numeric, impute+onehot categorical, LogisticRegression."
}
with open(META_PATH, "w") as f: json.dump(meta, f, indent=2)

# 2.3 simple checksum (integrity)
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()
print("SHA256(model):", sha256_file(MODEL_PATH))

SHA256(model): 87ca658b80f02fdac4e53cab6015cfb7e143ea494871414961a6de5987ba2a79


3) Load the model and predict (must match behavior!)

In [7]:
# In a fresh process/notebook you’d only need joblib + compatible sklearn
loaded = joblib.load(MODEL_PATH)
with open(META_PATH) as f: meta_loaded = json.load(f)
print("Loaded sklearn version:", meta_loaded["sklearn"], "| Current:", sklearn.__version__)

# 3.1 predict on the original Xte to verify identical outputs
same = np.allclose(loaded.predict_proba(Xte)[:,1], pipe.predict_proba(Xte)[:,1])
print("Bit‑for‑bit same proba on Xte? ->", bool(same))

# 3.2 predict on new raw records (dicts)
new_passengers = pd.DataFrame([
    {'pclass':3, 'sex':'male',   'age':22, 'sibsp':1, 'parch':0, 'fare':7.25, 'embarked':'S', 'class':'Third','who':'man',   'alone':False},
    {'pclass':1, 'sex':'female', 'age':38, 'sibsp':1, 'parch':0, 'fare':71.28,'embarked':'C', 'class':'First','who':'woman', 'alone':False},
    {'pclass':2, 'sex':'female', 'age':None,'sibsp':0,'parch':0, 'fare':13.0, 'embarked':None,'class':'Second','who':'woman','alone':True},
])
pred = loaded.predict(new_passengers)
proba = loaded.predict_proba(new_passengers)[:,1]
pd.DataFrame({"pred": pred, "proba": proba}).round(3)

Loaded sklearn version: 1.8.0 | Current: 1.8.0
Bit‑for‑bit same proba on Xte? -> True


,pred,proba
0,0,0.085
1,1,0.939
2,1,0.850


4) (Optional) Save a tiny model card (Markdown)

In [10]:
CARD_PATH = "MODEL_CARD.md"

# Notice the closing triple quotes at the very end
card = f"""# Titanic Survival Model (Logistic Regression)
**Date:** {meta['created_at']}
**Framework:** scikit-learn {meta['sklearn']}  
**Data:** seaborn.titanic — features: {', '.join(features)}; target: {target}

## Preprocessing
- Numeric: median imputation + StandardScaler
- Categorical: most-frequent imputation + OneHotEncoder(ignore unknown)

## Metrics (hold-out 20%)
- Accuracy: {accuracy_score(yte, pipe.predict(Xte)):.3f}
- ROC-AUC: {roc_auc_score(yte, pipe.predict_proba(Xte)[:,1]):.3f}

## Usage
"""

# Optional: Write the card content to the file
with open(CARD_PATH, "w", encoding="utf-8") as f:
    f.write(card)

print(f"Model card written to {CARD_PATH}")


Model card written to MODEL_CARD.md


5) (Optional) Save multiple versions and keep a registry row

In [12]:
# Pretend we trained another version (e.g., with different seed/hyperparams)
REGISTRY_PATH = "model_registry.csv"
row = {
    "version": "v1.0.0",
    "path": MODEL_PATH,
    "sha256": sha256_file(MODEL_PATH),
    "sklearn": sklearn.__version__,
    "created_at": meta["created_at"],
    "notes": "Baseline logistic pipeline"
}
try:
    reg = pd.read_csv(REGISTRY_PATH)
    reg = pd.concat([reg, pd.DataFrame([row])], ignore_index=True)
except FileNotFoundError:
    reg = pd.DataFrame([row])
reg.to_csv(REGISTRY_PATH, index=False)
reg.tail(3)

,version,path,sha256,sklearn,created_at,notes
0,v1.0.0,titanic_pipe.joblib,87ca658b80f02fdac4e53cab6015cfb7e143ea49487141...,1.8.0,2026-01-02T13:20:28.430497Z,Baseline logistic pipeline


6) (Optional) Regression mini‑demo (California Housing)
Exactly the same pattern works for regressors.

In [14]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
# Updated import: use root_mean_squared_error for RMSE
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

# Load data
data = fetch_california_housing(as_frame=True)
Xr = data.frame.drop(columns=['MedHouseVal'])
yr = data.frame['MedHouseVal']

# Build and fit pipeline
pipe_reg = Pipeline([
    ('prep', ColumnTransformer([('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                                                  ('sc',  StandardScaler())]),
                                Xr.columns.tolist())])),
    ('model', Ridge(alpha=3.0, random_state=0))
]).fit(Xr, yr)

# Save and load model
joblib.dump(pipe_reg, "california_ridge.joblib")
loaded_reg = joblib.load("california_ridge.joblib")
pred = loaded_reg.predict(Xr)

# Calculate metrics using the new function
print("R²:", round(r2_score(yr, pred), 3), "| RMSE:", round(root_mean_squared_error(yr, pred), 3))


R²: 0.606 | RMSE: 0.724


7) Common pitfalls & pro tips
Always save the entire Pipeline, not just the estimator—this preserves imputation/encoding/scaling.

Version pinning: record scikit-learn, numpy, pandas versions in metadata; loading with very different versions can fail.

Schema mismatches: new data must contain the same columns (names & dtypes). Keep features in metadata; validate before predicting.

Randomness: set random_state for deterministic artifacts.

Security: only load models you trust (joblib.load executes pickles). For untrusted artifacts, consider exporting ONNX or using a safer format.

Compression: joblib.dump(obj, path, compress=("xz", 3)) shrinks large models.